# Module 00: Setup & PyTorch Warm-Up

**Time:** ~20 minutes

Welcome to the TurboQuant deep-dive course! Before we dig into vector quantization,
we need to make sure your environment is ready and that you're comfortable with the
PyTorch operations we'll use throughout the course.

**What this notebook covers:**
1. Verify dependencies are installed
2. Tensor basics -- creating, inspecting, converting to/from numpy
3. Key operations: norms, random tensors, binary search, broadcasting, matrix multiply, reshaping
4. Exercise: normalize a batch of vectors to unit length

If you already know numpy, you're in great shape. PyTorch tensors are *very* similar
to numpy arrays, with two big additions: GPU support and automatic differentiation.
We'll focus on the parts that matter for quantization.

---
## 1. Setup -- Install & Verify Dependencies

In [ ]:
# Install dependencies if needed (safe to re-run)
%pip install -q torch numpy scipy plotly rich

In [ ]:
import torch
import numpy as np
import scipy
import plotly
import rich

from rich.console import Console
from rich.table import Table

console = Console()

# Show what we're working with
table = Table(title="Environment Check")
table.add_column("Package", style="bold")
table.add_column("Version")
table.add_column("Status")

for name, mod in [("torch", torch), ("numpy", np), ("scipy", scipy),
                   ("plotly", plotly), ("rich", rich)]:
    table.add_row(name, mod.__version__, "[green]OK[/green]")

# Check for CUDA (nice to have, not required)
cuda_status = "[green]Available[/green]" if torch.cuda.is_available() else "[yellow]Not available (CPU only -- that's fine)[/yellow]"
table.add_row("CUDA", cuda_status, "")

console.print(table)

In [ ]:
# Import our course helpers
import sys; sys.path.insert(0, "..")
from src.helpers import check_close, check_shape, section_header

section_header("Setup complete!", "All imports working. Let's learn some PyTorch.")

---
## 2. Tensor Basics

If you know numpy arrays, you already know 80% of PyTorch tensors.
Here's the mental model:

| Numpy | PyTorch | Notes |
|-------|---------|-------|
| `np.array([1,2,3])` | `torch.tensor([1,2,3])` | Create from a list |
| `arr.shape` | `t.shape` | Same attribute name! |
| `arr.dtype` | `t.dtype` | Similar, but prefixed: `torch.float32` |
| `np.zeros((3,4))` | `torch.zeros(3, 4)` | Note: no tuple needed in PyTorch |
| `arr + arr` | `t + t` | Element-wise ops work the same |
| N/A | `t.device` | PyTorch tracks CPU vs GPU |

Let's see these in action.

In [ ]:
section_header("Creating Tensors")

# From a Python list -- just like np.array()
t = torch.tensor([1.0, 2.0, 3.0])
console.print(f"From list:   {t}")
console.print(f"  shape:     {t.shape}")
console.print(f"  dtype:     {t.dtype}")
console.print(f"  device:    {t.device}")
console.print()

# 2D tensor -- think of it as a matrix (or a batch of vectors)
t2d = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
console.print(f"2D tensor:\n{t2d}")
console.print(f"  shape: {t2d.shape}  -- 3 rows, 2 columns")
console.print()

# Common constructors
console.print(f"zeros:  {torch.zeros(2, 3)}")
console.print(f"ones:   {torch.ones(2, 3)}")
console.print(f"arange: {torch.arange(0, 1, 0.25)}  -- like np.arange")

In [ ]:
section_header("Tensors <-> Numpy", "Converting back and forth")

# Numpy -> PyTorch
arr = np.array([10.0, 20.0, 30.0])
t_from_np = torch.from_numpy(arr)
console.print(f"numpy array:     {arr}  (type: {type(arr).__name__})")
console.print(f"as torch tensor: {t_from_np}  (type: {type(t_from_np).__name__})")
console.print()

# PyTorch -> Numpy
t = torch.tensor([1.0, 2.0, 3.0])
arr_from_t = t.numpy()
console.print(f"torch tensor:  {t}")
console.print(f"as numpy:      {arr_from_t}  (type: {type(arr_from_t).__name__})")
console.print()

# IMPORTANT: they share memory! Changing one changes the other.
arr_from_t[0] = 999.0
console.print(f"[yellow]After modifying the numpy array, the tensor changed too:[/yellow]")
console.print(f"  numpy: {arr_from_t}")
console.print(f"  torch: {t}")
console.print(f"  [dim]They share the same underlying memory. Use .clone() to avoid this.[/dim]")

In [ ]:
section_header("Data Types (dtypes)", "Quantization is all about precision, so dtypes matter.")

table = Table(title="Common PyTorch dtypes")
table.add_column("dtype", style="bold")
table.add_column("Bits")
table.add_column("Use case")
table.add_column("Example")

table.add_row("torch.float32", "32", "Default for most operations",
              str(torch.tensor(3.14)))
table.add_row("torch.float16", "16", "Half precision (saves memory)",
              str(torch.tensor(3.14, dtype=torch.float16)))
table.add_row("torch.int64", "64", "Default integer type",
              str(torch.tensor(42)))
table.add_row("torch.int8", "8", "Quantized values live here!",
              str(torch.tensor(42, dtype=torch.int8)))
table.add_row("torch.uint8", "8", "Unsigned (0-255), also for quantization",
              str(torch.tensor(200, dtype=torch.uint8)))

console.print(table)
console.print()
console.print("[dim]In this course, our embeddings start as float32 (32 bits per number).")
console.print("Quantization compresses them to far fewer bits -- that's the whole point![/dim]")

---
## 3. Key Operations for This Course

You don't need to know all of PyTorch. For the TurboQuant course, these are
the operations that come up again and again. Let's build intuition for each one.

### 3.1 `torch.randn()` -- Random Normal Tensors

We'll use this constantly to create synthetic embedding vectors for experiments.
`randn` draws from a standard normal distribution (mean=0, std=1).

In [ ]:
section_header("torch.randn()", "Random samples from N(0, 1)")

# A single random vector of dimension 5
v = torch.randn(5)
console.print(f"Single vector (dim=5): {v}")
console.print()

# A batch of vectors -- this is the typical shape we'll work with.
# Think of it as 1000 embedding vectors, each of dimension 128.
batch = torch.randn(1000, 128)
console.print(f"Batch shape: {batch.shape}  -- 1000 vectors, each 128-dimensional")
console.print(f"  mean:  {batch.mean():.4f}  (should be near 0)")
console.print(f"  std:   {batch.std():.4f}   (should be near 1)")
console.print()

# Reproducibility with manual_seed -- same seed gives same "random" numbers
torch.manual_seed(42)
a = torch.randn(3)
torch.manual_seed(42)
b = torch.randn(3)
console.print(f"With seed 42: {a}")
console.print(f"Again seed 42: {b}")
console.print(f"Identical? {torch.equal(a, b)}  -- [green]Seeds make experiments reproducible[/green]")

### 3.2 `torch.norm()` -- Vector Lengths

The L2 norm (Euclidean length) of a vector is fundamental to quantization.
Distortion is measured by how much norms change after quantization.

In [ ]:
section_header("torch.norm()", "Computing vector lengths (L2 norm)")

# Norm of a single vector: sqrt(3^2 + 4^2) = 5
v = torch.tensor([3.0, 4.0])
console.print(f"v = {v}")
console.print(f"||v|| = {torch.norm(v):.4f}  -- the classic 3-4-5 triangle")
console.print()

# Norm of each vector in a batch (this is the pattern we'll use most)
batch = torch.randn(5, 3)  # 5 vectors, each 3D
norms = torch.norm(batch, dim=-1)  # dim=-1 means "along the last dimension"
console.print(f"Batch shape: {batch.shape}")
console.print(f"Norms shape: {norms.shape}  -- one norm per vector")
console.print(f"Norms: {norms}")
console.print()

# The dim=-1 pattern:
#   If batch is (N, D), then norm(batch, dim=-1) gives shape (N,)
#   Each entry is the length of one D-dimensional vector.
console.print("[dim]dim=-1 means 'compute along the last axis' -- you'll see this everywhere.[/dim]")

### 3.3 `torch.searchsorted()` -- Binary Search

Quantization assigns each continuous value to its nearest discrete bucket.
`searchsorted` finds which bucket a value falls into, using binary search.
This is the same idea as `np.searchsorted` if you've used that.

In [ ]:
section_header("torch.searchsorted()", "Finding which bucket a value belongs to")

# Imagine we have bucket boundaries at these values:
boundaries = torch.tensor([-1.0, 0.0, 1.0, 2.0])

# Where does each value land?
values = torch.tensor([-2.0, -0.5, 0.5, 1.5, 3.0])
buckets = torch.searchsorted(boundaries, values)

console.print(f"Boundaries: {boundaries}")
console.print(f"Values:     {values}")
console.print(f"Buckets:    {buckets}")
console.print()

# Let's make this concrete:
for val, bucket in zip(values.tolist(), buckets.tolist()):
    if bucket == 0:
        desc = f"< {boundaries[0]:.1f}"
    elif bucket == len(boundaries):
        desc = f">= {boundaries[-1]:.1f}"
    else:
        desc = f"[{boundaries[bucket-1]:.1f}, {boundaries[bucket]:.1f})"
    console.print(f"  {val:5.1f} -> bucket {bucket}  (range: {desc})")

console.print()
console.print("[dim]In quantization, boundaries separate reconstruction levels.")
console.print("searchsorted tells us which level to use for each input value.[/dim]")

### 3.4 Broadcasting -- Operations on Different Shapes

Broadcasting lets you do math between tensors of different shapes without
explicitly copying data. This is *the same* as numpy broadcasting.

In [ ]:
section_header("Broadcasting", "Math with different-shaped tensors")

# Scalar + tensor: the scalar is broadcast to every element
t = torch.tensor([1.0, 2.0, 3.0])
console.print(f"t + 10 = {t + 10}  -- scalar broadcast to all elements")
console.print()

# Vector + matrix: the vector is broadcast across rows
# This is the pattern we'll use to subtract means, apply scales, etc.
matrix = torch.tensor([[1.0, 2.0, 3.0],
                        [4.0, 5.0, 6.0]])  # shape (2, 3)
row_means = matrix.mean(dim=-1, keepdim=True)   # shape (2, 1)
console.print(f"Matrix shape:     {matrix.shape}")
console.print(f"Row means shape:  {row_means.shape}")
console.print(f"Row means: {row_means.squeeze()}")
console.print()

centered = matrix - row_means  # (2,3) - (2,1) -> (2,3)
console.print(f"Centered matrix (each row now has mean ~0):\n{centered}")
console.print(f"Verify row means: {centered.mean(dim=-1)}  -- both near zero")
console.print()

# The key broadcasting rule:
# Dimensions are compared right-to-left.
# A dimension of size 1 is "stretched" to match the other tensor's size.
console.print("[dim]Broadcasting rule: size-1 dimensions stretch to match. Works right-to-left.[/dim]")

### 3.5 Matrix Multiply with `@`

The `@` operator does matrix multiplication. We'll use it for
rotation transforms (orthogonal matrices) that preserve distances.

In [ ]:
section_header("Matrix Multiply (@)", "The bread and butter of linear algebra")

# Basic matrix multiply
A = torch.tensor([[1.0, 2.0],
                   [3.0, 4.0]])  # (2, 2)
B = torch.tensor([[5.0, 6.0],
                   [7.0, 8.0]])  # (2, 2)

C = A @ B
console.print(f"A @ B =\n{C}")
console.print(f"Shapes: ({list(A.shape)}) @ ({list(B.shape)}) -> ({list(C.shape)})")
console.print()

# Batch of vectors times a matrix -- the typical pattern.
# (N, D) @ (D, D) -> (N, D)
# Think: "transform N vectors through a DxD matrix"
vectors = torch.randn(5, 3)  # 5 vectors, each 3D
transform = torch.randn(3, 3)  # 3x3 transform matrix
result = vectors @ transform
console.print(f"5 vectors (5,3) @ transform (3,3) = result {result.shape}")
console.print(f"  Each of the 5 vectors was multiplied by the 3x3 matrix.")
console.print()

# Dot product as matrix multiply
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([4.0, 5.0, 6.0])
console.print(f"Dot product: a @ b = {a @ b:.1f}  (same as 1*4 + 2*5 + 3*6 = 32)")

### 3.6 `.view()` and `.reshape()` -- Changing Tensor Shapes

Reshaping is how we reorganize data without changing the values.
In quantization, we'll reshape vectors into sub-vectors and back.

In [ ]:
section_header(".view() and .reshape()", "Same data, different shape")

# Start with a flat tensor
t = torch.arange(12, dtype=torch.float32)  # [0, 1, 2, ..., 11]
console.print(f"Original: {t}  shape={t.shape}")
console.print()

# Reshape into 2D
as_matrix = t.view(3, 4)  # 3 rows, 4 columns
console.print(f"view(3, 4):\n{as_matrix}")
console.print()

as_matrix2 = t.view(4, 3)  # 4 rows, 3 columns
console.print(f"view(4, 3):\n{as_matrix2}")
console.print()

# Use -1 to let PyTorch figure out one dimension
auto = t.view(2, -1)  # 2 rows, columns=12/2=6
console.print(f"view(2, -1): shape={auto.shape}  -- PyTorch computed the 6")
console.print()

# THE PATTERN WE'LL USE MOST:
# Split D-dimensional vectors into M sub-vectors of dimension D/M
D = 128     # embedding dimension
M = 8       # number of sub-vectors
d = D // M  # sub-vector dimension = 16

vectors = torch.randn(100, D)           # 100 vectors of dim 128
sub_vectors = vectors.view(100, M, d)   # 100 x 8 sub-vectors of dim 16

console.print(f"Splitting vectors into sub-vectors:")
console.print(f"  Original:     {vectors.shape}     -- 100 vectors, dim 128")
console.print(f"  Sub-vectors:  {sub_vectors.shape}  -- 100 vectors x 8 groups x dim 16")
console.print(f"  [dim]This is exactly how product quantization works![/dim]")
console.print()

# .view() vs .reshape(): view requires contiguous memory, reshape always works.
# Rule of thumb: try .view() first, switch to .reshape() if you get an error.
console.print("[dim]Tip: .view() and .reshape() do the same thing.")
console.print("Use .view() by default; switch to .reshape() if you hit a contiguity error.[/dim]")

---
## 4. Exercise: Normalize Vectors to Unit Length

Many quantization methods work with **unit vectors** (vectors with norm = 1).
This is your first hands-on exercise.

**Task:** Given a batch of random vectors, normalize each one to unit length.

A unit vector has the same direction as the original, but its length is exactly 1.
The formula is:

$$\hat{v} = \frac{v}{\|v\|}$$

**Steps:**
1. Compute the norm of each vector in the batch
2. Divide each vector by its norm (use broadcasting!)
3. Verify that all norms are now 1.0

In [ ]:
section_header("Exercise: Normalize to Unit Length")

# Here's your batch of vectors
torch.manual_seed(42)
vectors = torch.randn(64, 128)  # 64 vectors, each 128-dimensional

console.print(f"Input shape: {vectors.shape}")
console.print(f"Norms before: min={torch.norm(vectors, dim=-1).min():.4f}, "
              f"max={torch.norm(vectors, dim=-1).max():.4f}")
console.print(f"  (these are random, definitely not 1.0)")
console.print()

# ============================================================
# YOUR CODE HERE
#
# Step 1: Compute norms. Use torch.norm() with dim=-1 and keepdim=True
#          keepdim=True keeps the shape compatible for broadcasting.
# Step 2: Divide vectors by norms.
#
# Replace the line below with your solution:
# ============================================================

norms = torch.norm(vectors, dim=-1, keepdim=True)  # shape: (64, 1)
unit_vectors = vectors / norms                       # shape: (64, 128) -- broadcasting!

# ============================================================
# END YOUR CODE
# ============================================================

In [ ]:
# --- Verification ---

# Check 1: Shape should be the same as input
check_shape(unit_vectors, (64, 128), name="unit_vectors")

# Check 2: All norms should be 1.0
result_norms = torch.norm(unit_vectors, dim=-1)
expected_norms = torch.ones(64)
check_close(result_norms, expected_norms, name="All norms == 1.0")

# Check 3: Directions should be preserved (dot product of original and normalized
# should be positive -- they point the same way)
dot_products = (vectors * unit_vectors).sum(dim=-1)
all_positive = (dot_products > 0).all().item()
console.print(f"[{'green' if all_positive else 'red'}]Directions preserved: {all_positive}[/{'green' if all_positive else 'red'}]")

console.print()
console.print(f"Norms after: min={result_norms.min():.6f}, max={result_norms.max():.6f}")
console.print("[bold green]All norms are 1.0 -- these are unit vectors![/bold green]")

### Why does this matter for quantization?

Many quantization methods (including TurboQuant) separate the **direction** and
**magnitude** of a vector. The direction gets quantized with one method, the
magnitude (norm) with another. By normalizing first, we can focus on quantizing
directions on the unit sphere -- a much more structured problem.

You just did the first step of that pipeline!

---
## Recap

You now have:

- A working environment with all dependencies
- Comfort with PyTorch tensor creation, shapes, and dtypes
- Hands-on experience with the exact operations we'll use in quantization:
  - `torch.randn()` for synthetic data
  - `torch.norm()` for measuring vector lengths
  - `torch.searchsorted()` for bucket assignment
  - Broadcasting for batch operations
  - `@` for matrix multiply
  - `.view()` for reshaping into sub-vectors

**Next up:** Module 01 -- Scalar Quantization, where we'll compress individual
numbers into a small set of discrete levels.